> **💾 First: save your own copy** · Unless this notebook is already in your own Google Drive, Colab does not keep your changes. Go to **File → Save a copy in Drive** and work in the copy that opens in a new tab. You find it later in Google Drive, in *Colab Notebooks*.

![Cultural Data Analysis · Week 3: Structured Data and Machine Learning](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week3-banner.png)

# Data critique: a template for any table

This notebook goes with the ten questions of the [data critique](https://goto4711.github.io/cda-teaching-tools/workshops/week3/#critique). In the workshop, it loads the donation records of *Global Aid Alliance*; the case and its data sheet are on the workshop page. Most questions are answered by reading the data sheet, looking at the rows and discussing. The checks in Part 3 help you look and test a suspicion.

**In your group project**, use it on your own data: change `DATA_FILE` below, and the column names at the top of each check.

Run the cell below to load the data.

In [ ]:
#Keep cell
import pandas as pd

# A web address, or a file you uploaded to Colab (folder symbol on the left), e.g. 'my_data.csv'
DATA_FILE = 'https://raw.githubusercontent.com/goto4711/cdai/main/donor_records_export.csv'

if DATA_FILE.endswith(('.xlsx', '.xls')):
  data = pd.read_excel(DATA_FILE)
else:
  data = pd.read_csv(DATA_FILE)       # for a file separated by semicolons: pd.read_csv(DATA_FILE, sep=';')
print(len(data), 'rows,', len(data.columns), 'columns')
data.head()

---

## Your notes

For each answer, say what it rests on: *the data sheet says …*, *we saw in the data …*, or *we assume …*.

**The data: where it comes from**
1. Who made this data, for what purpose, and with what system or process? Whose question was it made to answer?
2. What does one row stand for, and what is outside the data: which people, periods or cases, and why?
3. Which categories did someone design? What can't be said in them?
4. What looks missing, wrong or inconsistent? Is it a mistake, or a trace of how the data was made?

**The label: what a model would predict**

5. What would the label be for the question being asked? Who defines it, and is it a proxy for what we actually care about?
6. Where do the features come from: measured, chosen by the people themselves, typed in by someone else, or computed by another system?

**The people: who is in it and who is affected**

7. Who are the people in the data? Did they know, did they agree, and did they agree to *this* use?
8. Which attributes are personal or sensitive, or could reveal something sensitive in combination? Could someone be identified?
9. Who could benefit, and who could be harmed or left out, if the analysis is used?

**Together**

10. Can this data answer the question at all? What would you need to know, or whom would you need to ask, before trusting it?

*Double-click here and write your answers below each question.*

---

## The checks

Each check says which questions it helps with. Change the column names in the first lines for your own data. Write your own checks in the empty cells at the end.

**Every column at a glance** (questions 3, 4, 6): its type, how many values are missing, how many different values it has, and a few examples.

In [ ]:
overview = pd.DataFrame({
  'type': data.dtypes.astype(str),
  'missing': data.isna().sum(),
  'missing %': (data.isna().mean() * 100).round(1),
  'different values': data.nunique(),
  'examples': [', '.join(str(v) for v in data[c].dropna().unique()[:4]) for c in data.columns],
})
overview

**The categories of one column** (question 3): which values did someone design, and how often does each occur? `(missing)` counts the empty cells.

In [ ]:
COLUMN = 'gender'

data[COLUMN].fillna('(missing)').value_counts()

**Two columns together** (questions 4, 6): does a column look different depending on another one, e.g. on how the data came in? Patterns like this are often traces of how the data was made.

In [ ]:
ROWS, COLUMNS = 'channel', 'gender'

pd.crosstab(data[ROWS].fillna('(missing)'), data[COLUMNS].fillna('(missing)'), margins=True, margins_name='total')

**Duplicates and IDs** (questions 2, 4, 8): are there rows that appear twice? Does one ID always stand for one person or thing, and does each person have one ID?

In [ ]:
ID, NAME = 'donor_id', 'donor_name'

print('Rows that appear more than once:', data.duplicated().sum())
names_per_id = data.groupby(ID)[NAME].nunique()
ids_per_name = data.groupby(NAME)[ID].nunique()
print(f'{ID}: {(names_per_id > 1).sum()} of {len(names_per_id)} have more than one {NAME}')
print(f'{NAME}: {(ids_per_name > 1).sum()} of {len(ids_per_name)} have more than one {ID}')
data[data[ID].isin(names_per_id[names_per_id > 1].index)].sort_values(ID).head(10)

**Time** (questions 2, 4): which period does the data cover, and which values of another column appear in which year? Dates that cannot be read are counted, not hidden.

In [ ]:
DATE, BY = 'donation_date', 'campaign'

dates = pd.to_datetime(data[DATE], errors='coerce')
print('From', dates.min().date(), 'to', dates.max().date(), '-', dates.isna().sum(), 'dates could not be read')
pd.crosstab(data[BY].fillna('(missing)'), dates.dt.year)

**Numbers** (question 4): the smallest, largest and typical values of the number columns. Look for values that cannot be right, or that look typed in by hand.

In [ ]:
data.describe()

**Your own checks**